## 7c. AI Agent as a Predictive Model
This section evaluates the AI agent on held-out real search queries and compares its predictive performance to the MNL model from Problem 1.

In [3]:
import pandas as pd
import numpy as np

real_df = pd.read_csv("data (1).csv")

choice_col_real = "booking_bool"

real_df["alt_id"] = real_df.groupby("srch_id").cumcount() + 1

features = [
    "prop_starrating",
    "prop_review_score",
    "prop_brand_bool",
    "prop_location_score",
    "prop_accesibility_score",
    "prop_log_historical_price",
    "price_usd",
    "promotion_flag"
]

In [4]:
rng = np.random.default_rng(42)

all_srch_ids = real_df["srch_id"].unique()

context_srch_ids = rng.choice(all_srch_ids, size=10, replace=False)

remaining_srch_ids = np.setdiff1d(all_srch_ids, context_srch_ids)

heldout_srch_ids = rng.choice(remaining_srch_ids, size=50, replace=False)

context_df = real_df[real_df["srch_id"].isin(context_srch_ids)].copy()
heldout_df = real_df[real_df["srch_id"].isin(heldout_srch_ids)].copy()

print("Context queries:", context_df["srch_id"].nunique())
print("Held-out queries:", heldout_df["srch_id"].nunique())
print("Held-out rows:", len(heldout_df))

Context queries: 10
Held-out queries: 50
Held-out rows: 868


In [5]:
def get_true_choice(group, choice_col):
    booked = group[group[choice_col] == 1]

    if len(booked) == 1:
        return int(booked.iloc[0]["alt_id"])
    elif len(booked) == 0:
        return "NO_PURCHASE"
    else:
        raise ValueError(f"Search {group['srch_id'].iloc[0]} has multiple bookings.")

true_choices = (
    heldout_df
    .groupby("srch_id")
    .apply(lambda g: get_true_choice(g, choice_col_real))
    .reset_index(name="true_choice")
)

true_choices.head()

/tmp/ipykernel_1098/165081072.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: get_true_choice(g, choice_col_real))


,srch_id,true_choice
0,43682,11
1,61359,1
2,65339,6
3,74418,3
4,88597,7


In [6]:
def summarize_query_for_prompt(group, include_answer=True):
    srch_id = group["srch_id"].iloc[0]

    context_cols = [
        "srch_booking_window",
        "srch_adults_count",
        "srch_children_count",
        "srch_room_count",
        "srch_saturday_night_bool",
    ]

    available_context_cols = [c for c in context_cols if c in group.columns]

    hotel_cols = ["alt_id"] + features
    available_hotel_cols = [c for c in hotel_cols if c in group.columns]

    context_text = group.iloc[0][available_context_cols].to_dict()
    hotel_table = group[available_hotel_cols].to_string(index=False)

    prompt_part = f"""
Search query {srch_id}
Customer/search context:
{context_text}

Hotel options:
{hotel_table}
""".strip()

    if include_answer:
        true_choice = get_true_choice(group, choice_col_real)
        prompt_part += f"\nObserved customer choice: {true_choice}"

    return prompt_part

In [9]:
context_examples = []

for srch_id, group in context_df.groupby("srch_id"):
    context_examples.append(summarize_query_for_prompt(group, include_answer=True))

heldout_prompts = []

for srch_id, group in heldout_df.groupby("srch_id"):
    heldout_prompts.append(summarize_query_for_prompt(group, include_answer=False))

full_prompt = f"""
You are predicting hotel booking choices.

Below are examples from real hotel search queries. Each example includes the customer/search context, the hotel alternatives shown, and the actual observed customer choice. The observed choice is either one hotel alt_id or NO_PURCHASE.

Context examples:
{"\n\n".join(context_examples)}

Now predict the customer choice for each held-out search query below. For each query, return exactly one prediction:
- one listed alt_id, or
- NO_PURCHASE

Return your answer as a CSV with columns:
srch_id,predicted_choice

Held-out queries:
{"\n\n".join(heldout_prompts)}
""".strip()

# print(full_prompt)
with open("ai_heldout_prompt.txt", "w", encoding="utf-8") as f:
    f.write(full_prompt)

## Comparison

In [23]:
ai_predictions = pd.read_csv("ai_heldout_predictions.csv")

ai_predictions["predicted_choice"] = ai_predictions["predicted_choice"].astype(str)
true_choices["true_choice"] = true_choices["true_choice"].astype(str)

ai_eval = true_choices.merge(ai_predictions, on="srch_id")

ai_eval["ai_correct"] = (
    ai_eval["true_choice"] == ai_eval["predicted_choice"]
)

ai_accuracy = ai_eval["ai_correct"].mean()

print("AI held-out query accuracy:", ai_accuracy)
# ai_eval

AI held-out query accuracy: 0.18


I use query-level exact-choice accuracy as the first metric. A prediction is counted as correct if it matches the exact booked hotel alternative, or correctly predicts `NO_PURCHASE` when the customer did not book.

In [12]:
# Human MNL normalized coefficients from Problem 1
human_beta = {
    "intercept": -1.9815321907864278,
    "prop_starrating": 0.4081249536151655,
    "prop_review_score": 0.10876096623704055,
    "prop_brand_bool": 0.22992269948768013,
    "prop_location_score": 0.02202632301274303,
    "prop_accesibility_score": 0.04344412341249515,
    "prop_log_historical_price": -0.06686945209512846,
    "price_usd": -1.3311099651462353,
    "promotion_flag": 0.45402977040295234
}

features = [
    "prop_starrating",
    "prop_review_score",
    "prop_brand_bool",
    "prop_location_score",
    "prop_accesibility_score",
    "prop_log_historical_price",
    "price_usd",
    "promotion_flag"
]

# Use non-held-out data to compute normalization values
train_df = real_df[~real_df["srch_id"].isin(heldout_srch_ids)].copy()

feature_means = train_df[features].mean()
feature_stds = train_df[features].std().replace(0, 1)

def mnl_predict_query(group, beta, feature_means, feature_stds):
    g = group.copy()

    g[features] = g[features].replace([np.inf, -np.inf], np.nan)
    g[features] = g[features].fillna(feature_means)

    X = (g[features] - feature_means) / feature_stds

    u = beta["intercept"]
    for f in features:
        u += beta[f] * X[f]

    max_idx = u.idxmax()
    max_u = u.loc[max_idx]

    # outside option utility is normalized to 0
    if max_u <= 0:
        return "NO_PURCHASE"
    else:
        return str(int(g.loc[max_idx, "alt_id"]))

mnl_predictions = []

for srch_id, group in heldout_df.groupby("srch_id"):
    pred = mnl_predict_query(group, human_beta, feature_means, feature_stds)
    mnl_predictions.append({
        "srch_id": srch_id,
        "mnl_predicted_choice": pred
    })

mnl_predictions = pd.DataFrame(mnl_predictions)

# Merge AI, MNL, and true outcomes
eval_df = (
    true_choices
    .merge(ai_predictions, on="srch_id")
    .merge(mnl_predictions, on="srch_id")
)

eval_df["true_choice"] = eval_df["true_choice"].astype(str)
eval_df["predicted_choice"] = eval_df["predicted_choice"].astype(str)
eval_df["mnl_predicted_choice"] = eval_df["mnl_predicted_choice"].astype(str)

eval_df["ai_correct"] = eval_df["true_choice"] == eval_df["predicted_choice"]
eval_df["mnl_correct"] = eval_df["true_choice"] == eval_df["mnl_predicted_choice"]

accuracy_table = pd.DataFrame({
    "Model": ["AI agent", "MNL"],
    "Held-out query accuracy": [
        eval_df["ai_correct"].mean(),
        eval_df["mnl_correct"].mean()
    ],
    "Correct predictions": [
        eval_df["ai_correct"].sum(),
        eval_df["mnl_correct"].sum()
    ],
    "Held-out queries": [
        len(eval_df),
        len(eval_df)
    ]
})

accuracy_table

,Model,Held-out query accuracy,Correct predictions,Held-out queries
0,AI agent,0.18,9,50
1,MNL,0.22,11,50


Exact-choice accuracy is a strict metric because each search query contains many hotel alternatives. A model only receives credit if it predicts the exact booked hotel or correctly predicts no-purchase. On this metric, the MNL model slightly outperforms the AI agent: MNL predicts 11 of 50 held-out choices correctly, while the AI predicts 9 of 50 correctly.

In [13]:
import pandas as pd
import numpy as np

features_to_compare = [
    "price_usd",
    "prop_starrating",
    "prop_review_score",
    "prop_location_score",
    "prop_brand_bool",
    "promotion_flag"
]

# Make sure choice columns are strings
true_choices["true_choice"] = true_choices["true_choice"].astype(str)
ai_predictions["predicted_choice"] = ai_predictions["predicted_choice"].astype(str)
mnl_predictions["mnl_predicted_choice"] = mnl_predictions["mnl_predicted_choice"].astype(str)

def get_chosen_rows(pred_df, pred_col, label):
    rows = []

    for _, row in pred_df.iterrows():
        srch_id = row["srch_id"]
        choice = str(row[pred_col])

        if choice == "NO_PURCHASE":
            rows.append({
                "srch_id": srch_id,
                "model": label,
                "choice": "NO_PURCHASE",
                "purchased": 0,
                **{f: np.nan for f in features_to_compare}
            })
        else:
            alt_id = int(choice)
            chosen = heldout_df[
                (heldout_df["srch_id"] == srch_id) &
                (heldout_df["alt_id"] == alt_id)
            ]

            if len(chosen) == 1:
                chosen = chosen.iloc[0]
                rows.append({
                    "srch_id": srch_id,
                    "model": label,
                    "choice": choice,
                    "purchased": 1,
                    **{f: chosen[f] for f in features_to_compare}
                })

    return pd.DataFrame(rows)

observed_behavior = get_chosen_rows(
    true_choices.rename(columns={"true_choice": "choice"}),
    "choice",
    "Observed human data"
)

ai_behavior = get_chosen_rows(
    ai_predictions,
    "predicted_choice",
    "AI agent"
)

mnl_behavior = get_chosen_rows(
    mnl_predictions,
    "mnl_predicted_choice",
    "MNL"
)

behavior_rows = pd.concat(
    [observed_behavior, ai_behavior, mnl_behavior],
    ignore_index=True
)

summary = (
    behavior_rows
    .groupby("model")
    .agg(
        purchase_rate=("purchased", "mean"),
        no_purchase_rate=("purchased", lambda x: 1 - x.mean()),
        avg_chosen_price=("price_usd", "mean"),
        avg_chosen_star_rating=("prop_starrating", "mean"),
        avg_chosen_review_score=("prop_review_score", "mean"),
        avg_chosen_location_score=("prop_location_score", "mean"),
        share_chosen_brand=("prop_brand_bool", "mean"),
        share_chosen_promotion=("promotion_flag", "mean")
    )
    .reset_index()
)

summary

,model,purchase_rate,no_purchase_rate,avg_chosen_price,avg_chosen_star_rating,avg_chosen_review_score,avg_chosen_location_score,share_chosen_brand,share_chosen_promotion
0,AI agent,0.82,0.18,136.829268,3.707317,4.926829,3.000000,0.902439,0.317073
1,MNL,0.08,0.92,88.250000,4.250000,4.000000,2.750000,1.000000,1.000000
2,Observed human data,0.74,0.26,115.081081,3.243243,4.135135,2.594595,0.783784,0.243243


### Behavioral comparison using hard predictions

Exact-choice accuracy is a strict metric because each search query contains many hotel alternatives, and a model only receives credit if it predicts the exact booked hotel or correctly predicts no-purchase. To complement exact accuracy, I also compare the aggregate behavior implied by each model to the behavior observed in the held-out data.

The observed purchase rate in the held-out sample is 74%. The AI agent predicts a purchase rate of 82%, which is relatively close to the observed rate. In contrast, the MNL hard-choice prediction produces a purchase rate of only 8%, implying no-purchase in 92% of held-out queries. This suggests that the MNL hard-choice predictions are very conservative on this held-out sample when forced to select only the single highest-probability outcome.

The AI agent also produces chosen hotels with somewhat higher average price, star rating, review score, and location score than the observed human choices. This suggests that the AI tends to over-select hotels that look clearly attractive on observable attributes. Still, its aggregate purchase/no-purchase behavior is closer to the observed data than the MNL hard-choice predictions.

In [25]:
def mnl_probabilities_for_query(group, beta, feature_means, feature_stds):
    g = group.copy()

    g[features] = g[features].replace([np.inf, -np.inf], np.nan)
    g[features] = g[features].fillna(feature_means)

    X = (g[features] - feature_means) / feature_stds

    u = beta["intercept"]
    for f in features:
        u += beta[f] * X[f]

    # numerical stability
    max_u = max(0, np.max(u))
    exp_u = np.exp(u - max_u)
    exp_outside = np.exp(0 - max_u)

    denom = exp_outside + exp_u.sum()

    hotel_probs = exp_u / denom
    no_purchase_prob = exp_outside / denom

    return hotel_probs, no_purchase_prob



# Aggregate MNL probability-implied behavior across all held-out queries
mnl_prob_rows = []

for srch_id, group in heldout_df.groupby("srch_id"):
    hotel_probs, no_purchase_prob = mnl_probabilities_for_query(
        group,
        human_beta,
        feature_means,
        feature_stds
    )

    temp = group.copy()
    temp["mnl_prob"] = hotel_probs
    temp["no_purchase_prob"] = no_purchase_prob

    mnl_prob_rows.append(temp)

mnl_prob_df = pd.concat(mnl_prob_rows, ignore_index=True)

total_purchase_prob = mnl_prob_df["mnl_prob"].sum()
num_queries = heldout_df["srch_id"].nunique()

mnl_expected_summary = pd.DataFrame({
    "model": ["MNL probability-implied"],
    "purchase_rate": [total_purchase_prob / num_queries],
    "no_purchase_rate": [1 - total_purchase_prob / num_queries],
    "avg_chosen_price": [
        (mnl_prob_df["mnl_prob"] * mnl_prob_df["price_usd"]).sum() / total_purchase_prob
    ],
    "avg_chosen_star_rating": [
        (mnl_prob_df["mnl_prob"] * mnl_prob_df["prop_starrating"]).sum() / total_purchase_prob
    ],
    "avg_chosen_review_score": [
        (mnl_prob_df["mnl_prob"] * mnl_prob_df["prop_review_score"]).sum() / total_purchase_prob
    ],
    "avg_chosen_location_score": [
        (mnl_prob_df["mnl_prob"] * mnl_prob_df["prop_location_score"]).sum() / total_purchase_prob
    ],
    "share_chosen_brand": [
        (mnl_prob_df["mnl_prob"] * mnl_prob_df["prop_brand_bool"]).sum() / total_purchase_prob
    ],
    "share_chosen_promotion": [
        (mnl_prob_df["mnl_prob"] * mnl_prob_df["promotion_flag"]).sum() / total_purchase_prob
    ]
})

behavior_summary_with_expected_mnl = pd.concat(
    [summary, mnl_expected_summary],
    ignore_index=True
)

behavior_summary_with_expected_mnl

,model,purchase_rate,no_purchase_rate,avg_chosen_price,avg_chosen_star_rating,avg_chosen_review_score,avg_chosen_location_score,share_chosen_brand,share_chosen_promotion
0,AI agent,0.820000,0.180000,136.829268,3.707317,4.926829,3.000000,0.902439,0.317073
1,MNL,0.080000,0.920000,88.250000,4.250000,4.000000,2.750000,1.000000,1.000000
2,Observed human data,0.740000,0.260000,115.081081,3.243243,4.135135,2.594595,0.783784,0.243243
3,MNL probability-implied,0.656018,0.343982,123.627156,3.229970,3.994446,2.375955,0.792406,0.316118


### MNL probability-implied behavior

The MNL model naturally outputs probabilities for each hotel alternative and for the no-purchase option. The hard-choice prediction discards most of this information by selecting only the single highest-probability outcome. To use the full probability distribution, I also summarize the MNL-implied behavior in expectation.

For each query, the MNL-implied purchase probability is the sum of the probabilities assigned to all hotel alternatives. The expected chosen-hotel attributes are computed as probability-weighted averages of hotel attributes, conditional on purchase.

This probability-implied version gives a more reasonable picture of the MNL model. The hard-choice MNL predicts purchase in only 8% of held-out queries, while the probability-implied MNL purchase rate is about 65.6%, much closer to the observed 74%. Its expected chosen-hotel price, star rating, review score, brand share, and promotion share are also much closer to observed behavior than the hard-choice MNL summary. This shows why relying only on the highest-probability choice can lose useful information in a discrete choice model.

### Interpretation

The exact-choice accuracy metric slightly favors the MNL model: MNL correctly predicts 11 out of 50 held-out choices, while the AI agent correctly predicts 9 out of 50. However, exact-choice accuracy is a high bar in this setting because many hotel alternatives are similar and customer choices are noisy.

The behavioral metrics provide a more nuanced comparison. The AI agent better matches the observed purchase/no-purchase rate than the MNL hard-choice rule, while the MNL probability-implied summary shows that the full MNL probability distribution is much closer to observed behavior than its hard prediction alone. Overall, the AI agent captures some broad behavioral patterns, but the MNL model remains useful because it provides a full probability distribution over choices rather than only a single prediction. Because the held-out set contains only 50 search queries, these results should be interpreted as illustrative rather than definitive. A larger held-out set or repeated random splits would provide a more stable comparison.